# registry

> Defining hyperelements and collecting their styles and behaviors

In [ ]:
#| default_exp registry

A UI component needs HTML markup, CSS styling, and sometimes browser behavior. FastHTML already gives us the markup constructors. Here we build a registry to hold the styling and behaviors, associate them with their constructor, and collect everything a page needs into one stylesheet and one script. The same registry holds application styling that belongs to no component, such as page layout.

In [ ]:
import inspect
from collections.abc import Mapping
from fastcore.utils import *
import fasthtml.components

In [ ]:
from fasthtml.common import Label, Input, Button, to_xml
from fastcore.test import *

## Rules

Styling and behavior have the same shape. Each pairs a *selector* that picks elements with a *payload* saying what happens to them: CSS declarations for styling, an event and handler for behavior. Each must also reach the page inside one aggregated asset (one stylesheet, one script). So we register them together: a named rule holds CSS text and behavior records, filed under a [cascade layer](https://developer.mozilla.org/en-US/docs/Web/CSS/@layer).

The layer list is fixed and ordered, and each layer beats the ones before it for normal declarations, regardless of specificity (`!important` reverses the order, one reason the library never uses it), so an application can restyle a built-in component without touching its definition.


In [ ]:
LAYERS = ('reset','tokens','base','layout','components','app','utilities')

Behaviors are a dict mapping `(selector, event)` keys to JavaScript handler bodies. Each handler runs with `el` bound to the matched element and `e` to the event. A key may include a truthy third item to attach the handler in the capture phase, needed for events like `toggle` that don't bubble. An entry may also declare a root selector `sel`; `&` in a behavior selector then stands for it, as in CSS nesting.

In [ ]:
def _behaviors(on, sel):
    "Behavior records with `&` replaced by `sel`, wrapped in `:is()` so a selector list stays grouped"
    res = []
    for (selector, event, *capture), js in (on or {}).items():
        if '&' in selector:
            if sel is None: raise ValueError(f'`&` in {selector!r} needs sel')
            selector = selector.replace('&', f':is({sel})')
        res.append(dict(sel=selector, ev=event, js=js, cap=bool(capture and capture[0])))
    return res

A rule is keyed by a name, or by `owner.name` when it belongs to a hyperelement, and the key is its identity: `Rule` checks the layer, expands the behaviors, and reads the owner off the key. A rule that declares a variant records the attribute:


In [ ]:
class Rule:
    "A named CSS and behavior definition in a cascade layer"
    def __init__(
        self,
        key,          # Registry key: a name, or `owner.name` for a hyperelement's rule
        css='',       # Complete CSS, or a no-argument function returning it
        *,
        on=None,      # Mapping of (selector, event[, capture]) to JavaScript handler bodies
        sel=None,     # Root selector substituted for `&` in behaviors
        layer='app',  # One of LAYERS
        doc='',       # Description of the rule
        variant=None, # Attribute selecting a declared variant
    ):
        if layer not in LAYERS: raise ValueError(f'Unknown layer: {layer}')
        store_attr('key,css,sel,layer,doc,variant')
        self.on = _behaviors(on, sel)

    @property
    def owner(self): return self.key.split('.', 1)[0] if '.' in self.key else None

    @property
    def name(self): return self.key.split('.', 1)[-1]

    @property
    def compiled_css(self):
        "Evaluate callable CSS and return the stylesheet text"
        return (self.css() if callable(self.css) else self.css).strip()

    def __repr__(self):
        sel = f' {self.sel}' if self.sel else ''
        doc = f': {self.doc}' if self.doc else ''
        return f'{self.key} [{self.layer}]{sel}{doc}'

The registry keeps every rule in one dict, keyed by name, or by `owner.name` for a hyperelement's rules. Registering a key again updates the existing rule in place, so a reference taken earlier stays current and the rule keeps its position in the stylesheet. A component's rule can be replaced by its full key as readily as through the component. Every registration and removal advances `version`, which the asset builders watch to rebuild only after a change. Layers are live views onto the same dict:


In [ ]:
class RuleView(Mapping):
    "A live, read-only view of one layer or one hyperelement's rules"
    def __init__(self, registry, *, layer=None, owner=None): store_attr('registry,layer,owner')
    def _matches(self, r): return (self.layer is None or r.layer==self.layer) and (self.owner is None or r.owner==self.owner)
    def __iter__(self): return (r.name if self.owner is not None else k for k,r in self.registry.rules.items() if self._matches(r))
    def __len__(self): return sum(1 for _ in self)
    def __getitem__(self, name):
        key = f'{self.owner}.{name}' if self.owner is not None else name
        r = self.registry.rules[key]
        if not self._matches(r): raise KeyError(name)
        return r
    def __repr__(self): return '\n'.join(repr(r) for r in self.values()) or '(no rules)'

class Registry:
    "Named rules and hyperelements, shared by the asset builders"
    def __init__(self):
        self.rules, self.elements, self.version = {}, {}, 0
        self.layers = {name: RuleView(self, layer=name) for name in LAYERS}

    def rule(
        self,
        key,          # Registry key: a name, or `owner.name` for a hyperelement's rule
        css='',       # Complete CSS, or a no-argument function returning it
        *,
        on=None,      # Mapping of (selector, event[, capture]) to JavaScript handler bodies
        sel=None,     # Root selector substituted for `&` in behaviors; does not scope CSS
        layer='app',  # One of LAYERS
        doc='',       # Description of the rule
        variant=None, # Attribute selecting a declared variant
    ):
        "Register a rule, replacing an existing definition in place"
        r = Rule(key, css, on=on, sel=sel, layer=layer, doc=doc, variant=variant)
        if r.key in self.rules: self.rules[r.key].__dict__.update(vars(r))
        else: self.rules[r.key] = r
        self.version += 1
        return self.rules[r.key]

    def remove(self, key):
        "Remove and return the rule at its full registry key"
        r = self.rules.pop(key)
        self.version += 1
        return r

    def __repr__(self): return f'Registry({len(self.rules)} rules, {len(self.elements)} elements)'

    def _repr_markdown_(self):
        layers = '\n'.join(f'- `{name}`: {len(rules)} rules' for name,rules in self.layers.items())
        elements = ', '.join(f'`{k}`' for k in self.elements) or 'none'
        return f'{self!r}\n\nLayers:\n{layers}\n\nElements: {elements}'

reg = Registry()

The simplest entry is bare CSS with no constructor; for example, page structure like a centered column. `app` is the default layer, since an application is the usual author of an entry like this:


In [ ]:
reg.rule('app-page', 'main { max-width: 65ch; margin-inline: auto; padding: 1rem; }', doc='A centered single-column page.')

app-page [app]: A centered single-column page.

A behavior with no constructor of its own works the same way. Here's a `copy` attribute: put it on any control, and a click copies the attribute's value, or failing that the element's text, to the clipboard. The attribute is the natural root selector, so we declare it as `sel` and write the behavior against `&`. What the entry stores is the expanded record:

In [ ]:
reg.rule('copy', sel='[copy]', on={('&', 'click'): r"navigator.clipboard.writeText(el.getAttribute('copy') || el.textContent)"},
    doc='A control carrying `copy` copies its value, or its text, to the clipboard.')
reg.rules['copy'].on

[{'sel': ':is([copy])',
  'ev': 'click',
  'js': "navigator.clipboard.writeText(el.getAttribute('copy') || el.textContent)",
  'cap': False}]

In [ ]:
with expect_fail(ValueError): reg.rule('app-links', 'a { color: teal; }', layer='appp')
with expect_fail(ValueError): reg.rule('paste', on={('&', 'click'): ''})

## Building the stylesheet

We start with a bare `@layer` statement listing the layer names to declare their precedence, then emit each layer's rules as a block in registration order, each headed by a comment naming its rule. A rule's CSS may be a function, called here, for rules that depend on registrations made after their own:


In [ ]:
@patch
def build_css(self:Registry):
    "Assemble the registry's CSS in cascade-layer order"
    out = [f"@layer {', '.join(LAYERS)};"]
    for layer, rules in self.layers.items():
        parts = [f'/* {r.key} */\n{css}' for r in rules.values() if (css := r.compiled_css)]
        if parts: out.append(f'@layer {layer} {{\n' + '\n'.join(parts) + '\n}')
    return '\n\n'.join(out) + '\n'

In [ ]:
print(reg.build_css())

@layer reset, tokens, base, layout, components, app, utilities;

@layer app {
/* app-page */
main { max-width: 65ch; margin-inline: auto; padding: 1rem; }
}



## Hyperelements

A component is a constructor (i.e. a standard FastTag or a Python function bundling several together) plus rules. A `Hyperelement` binds the two: calling it calls the constructor, and beside it sit the root selector and a live view of its rules:


In [ ]:
def _nest(sel, css): return f'{sel} {{\n{css.strip()}\n}}' if css.strip() else ''
def _generic(*children, **attrs): ...

class Hyperelement:
    "A callable constructor linked to its registered rules and behaviors"
    def __init__(self, registry, name, fn, sel, doc='', generated=False):
        self.registry, self.__name__, self.fn, self.sel = registry, name, fn, sel
        self.__doc__, self.generated = doc, generated
        self.rules = RuleView(registry, owner=name)

    def __call__(self, *c, **attrs): return self.fn(*c, **attrs)

    @property
    def __signature__(self): return inspect.signature(_generic if self.generated else self.fn)

    @property
    def variants(self):
        "Declared variant attributes and their rules"
        return {r.variant: r for r in self.rules.values() if r.variant is not None}

    @property
    def behaviors(self):
        "Behavior records declared by this hyperelement's rules"
        return [b for r in self.rules.values() for b in r.on]

    def __repr__(self): return f'{self.__name__} [{self.sel}]'

    def _repr_markdown_(self):
        name = self.__name__
        sig = str(self.__signature__)
        parts = [f'### `{name}{sig}`', self.__doc__, f'Root selector: `{self.sel}`.']
        if self.variants:
            variants = [f'- `{attr}`: {r.doc}' for attr,r in self.variants.items()]
            parts.append('Declared variants:\n' + '\n'.join(variants))
        rules = [f'- `{k}` ({r.layer})' + (f': {r.doc}' if r.doc and r.doc != self.__doc__ else '') for k,r in self.rules.items() if r.variant is None]
        parts.append('Rules:\n' + '\n'.join(rules))
        return '\n\n'.join(p for p in parts if p)

An application's additions to a component are its `rule` and `variant` methods. Both nest the CSS they are given, `rule` under the root selector and `variant` under an attribute selector, and register it with the hyperelement as owner. `variant` also records the attribute, which is how a variant is declared: writing its CSS is the declaration:

In [ ]:
@patch
def rule(
    self:Hyperelement,
    name,         # Local rule name; registering it again replaces that rule
    css='',       # CSS declarations and nested rules, scoped to the root selector
    *,
    on=None,      # Mapping of (selector, event[, capture]) to handler bodies; `&` is the root selector
    layer='app',  # One of LAYERS
    doc='',       # Description of the customization
):
    "Register CSS and behaviors scoped to this hyperelement"
    return self.registry.rule(f'{self.__name__}.{name}', _nest(self.sel, css), on=on, sel=self.sel, layer=layer, doc=doc)

@patch
def variant(
    self:Hyperelement,
    name,         # Attribute selecting the variant; also its local rule name
    css='',       # CSS declarations and nested rules, scoped to the variant
    *,
    on=None,      # Mapping of (selector, event[, capture]) to handler bodies; `&` is the variant selector
    layer='app',  # One of LAYERS
    doc='',       # Description shown with the variant
):
    "Declare an attribute variant without changing the constructor"
    if name=='base': raise ValueError('base is reserved for the constructor definition')
    sel = f':is({self.sel})[{name}]'
    return self.registry.rule(f'{self.__name__}.{name}', _nest(sel, css), on=on, sel=sel, layer=layer, doc=doc, variant=name)

`hypelt` registers the pair. It takes a FastHTML tag name, or decorates a handwritten constructor, and stores the component's complete stylesheet as a rule called `base`. The module-level `hypelt` registers with `reg`:


In [ ]:
@patch
def hypelt(
    self:Registry,
    name=None,   # FastHTML tag name or a constructor; omit to use as a decorator
    *,
    sel=None,    # Root selector; required for a handwritten constructor
    css='',      # Complete CSS, or a no-argument function returning it; stored as the base rule
    on=None,     # Mapping of (selector, event[, capture]) to JavaScript handler bodies
    layer='app', # One of LAYERS; built-ins use components, layout, or base
    doc=None,    # Description; defaults to a handwritten constructor's docstring
):
    "Register a constructor and its base rule, returning a callable hyperelement"
    if name is None: return partial(self.hypelt, sel=sel, css=css, on=on, layer=layer, doc=doc)
    generated = isinstance(name, str)
    if generated:
        fn = getattr(fasthtml.components, name)
        if sel is None: sel = fn().tag
    else:
        fn, name = name, name.__name__
        if sel is None: raise ValueError('A handwritten constructor needs sel')
        if doc is None: doc = fn.__doc__
    if name in self.elements and self.elements[name].sel != sel:
        raise ValueError(f'{name} is registered with root selector {self.elements[name].sel!r}, which its rules and variants target; register a new hyperelement for a new root')
    self.rule(f'{name}.base', css, on=on, sel=sel, layer=layer, doc=doc or '')
    obj = Hyperelement(self, name, fn, sel, doc or '', generated)
    if name in self.elements: self.elements[name].__dict__.update(vars(obj))
    else: self.elements[name] = obj
    return self.elements[name]

hypelt = reg.hypelt

As an example component, let's make a dismissible notice — a box announcing something like "changes saved", with a button to close it. HTML has no element for that, so it gets a custom tag, `notice-box`, and its behavior is that a click on any child marked `dismiss` removes the notice:

In [ ]:
notice_css = r"""
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
"""
NoticeBox = hypelt('NoticeBox', css=notice_css, on={('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"},
    layer='components', doc='A dismissible notice.')
to_xml(NoticeBox('Changes saved.', Button('Dismiss', dismiss=True)))

'<notice-box>Changes saved.<button dismiss>Dismiss</button></notice-box>'

FastHTML generated the constructor from the CamelCase name, and since calling it with no arguments gives an empty element, `hypelt` read the root selector straight off the tag. The hyperelement's view and the registry hold the same rule object:


In [ ]:
test_eq(NoticeBox().tag, NoticeBox.sel)
assert NoticeBox.rules['base'] is reg.rules['NoticeBox.base']
NoticeBox

### `NoticeBox(*children, **attrs)`

A dismissible notice.

Root selector: `notice-box`.

Rules:
- `base` (components)

The second case is a handwritten constructor. Consider a labelled field. It wants native `label` and `input` elements, with the input wrapped in its label so the two are associated without an id. Our function, `Field`, needs arguments, so there's no empty call to read a selector from — we mark the label with a `field` attribute and supply it as `sel`, using the decorator form:

In [ ]:
field_css = r"""
label[field] {
  display: grid;
  gap: .5rem;
}
"""

@hypelt(sel='label[field]', css=field_css)
def Field(label, **attrs):
    "A labelled input"
    return Label(label, Input(**attrs), field=True)

to_xml(Field('Email', name='email', type='email'))

'<label field>Email  <input name="email" type="email">\n</label>'

## Customization

There are two ways to change how a component looks or behaves. An override registers separate CSS in the `app` layer, leaving the component's entry alone; redefinition registers the same name again, and the new entry replaces the old one completely.

### Overrides

Say a developer wants rounder notices in their application. A named rule on the hyperelement lands in the `app` layer by default, so it wins over the component's own CSS (in the `components` layer):

In [ ]:
NoticeBox.rule('corners', 'border-radius: 1rem;', doc='Rounder notices in this application.')


NoticeBox.corners [app] notice-box: Rounder notices in this application.

When an attribute should select a treatment, declare it as a variant. Its description then appears on the hyperelement, and the same constructor accepts the attribute:


In [ ]:
NoticeBox.variant('quiet', 'background: transparent; border-color: transparent;', doc='An unfilled notice without a visible border.')


NoticeBox.quiet [app] :is(notice-box)[quiet]: An unfilled notice without a visible border.

In [ ]:
test_eq(list(NoticeBox.variants), ['quiet'])
to_xml(NoticeBox('Saved quietly.', quiet=True))

'<notice-box quiet>Saved quietly.</notice-box>'

A change confined to part of a page is an application rule with its own selector. It adds no component and touches nothing in the notice's definition, and the layer view shows it beside the others. It reaches every notice inside an editor without appearing in `NoticeBox.rules`, which lists the rules registered on the component rather than every rule that can style it:


In [ ]:
reg.rule('editor-notices', '[editor] notice-box { border-radius: 0; }', doc='Square notices inside the editor.')
reg.layers['app']

app-page [app]: A centered single-column page.
copy [app] [copy]: A control carrying `copy` copies its value, or its text, to the clipboard.
Field.base [app] label[field]: A labelled input
NoticeBox.corners [app] notice-box: Rounder notices in this application.
NoticeBox.quiet [app] :is(notice-box)[quiet]: An unfilled notice without a visible border.
editor-notices [app]: Square notices inside the editor.

### Overwrites

Registering a hyperelement again updates its constructor and base rule in place. References held elsewhere point at the current definition, and the application's rules and declared variants on it stay registered. Omitted CSS or behaviors are cleared rather than merged, since the new call is the whole definition. The root selector stays: the component's rules and variants were written against it, so a redefinition with a different root is rejected. Here the notice becomes persistent:


In [ ]:
original, base = NoticeBox, NoticeBox.rules['base']
NoticeBox = hypelt('NoticeBox', css=notice_css.replace('8px', '12px'), layer='components', doc='A persistent notice.')
test_is(NoticeBox, original)
test_is(NoticeBox.rules['base'], base)
assert '12px' in base.css
assert 'corners' in NoticeBox.rules and 'quiet' in NoticeBox.variants
with expect_fail(ValueError): hypelt('NoticeBox', sel='notice-box[persistent]', css=notice_css, layer='components')
NoticeBox

### `NoticeBox(*children, **attrs)`

A persistent notice.

Root selector: `notice-box`.

Declared variants:
- `quiet`: An unfilled notice without a visible border.

Rules:
- `base` (components)
- `corners` (app): Rounder notices in this application.

The base rule now declares no behavior. Removing the application override uses its full key, and a layer view taken earlier reflects the removal:


In [ ]:
test_eq(NoticeBox.behaviors, [])
app_rules = reg.layers['app']
v = reg.version
reg.remove('NoticeBox.corners')

NoticeBox.corners [app] notice-box: Rounder notices in this application.

In [ ]:
assert 'NoticeBox.corners' not in app_rules
test_eq(reg.version, v+1)
app_rules

app-page [app]: A centered single-column page.
copy [app] [copy]: A control carrying `copy` copies its value, or its text, to the clipboard.
Field.base [app] label[field]: A labelled input
NoticeBox.quiet [app] :is(notice-box)[quiet]: An unfilled notice without a visible border.
editor-notices [app]: Square notices inside the editor.

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()